# 03 — Model Training
## Loan Default Prediction System

**Purpose:** Perform a stratified train/test split, compare candidate algorithms (Logistic Regression, Random Forest, XGBoost) under identical cross-validation folds, hyperparameter-tune the best performer, and save the final `Pipeline` (preprocessing + model) to `ml/models/loan_pipeline.pkl`.

**Input:** `ml/data/raw/Loan_default.csv`
**Output:** `ml/models/loan_pipeline.pkl`, `ml/data/processed/train_set.csv`, `ml/data/processed/test_set.csv`, `ml/reports/model_comparison_report.md`

This notebook is a thin, exploratory wrapper around `ml/src/train_pipeline.py` — every function called here is the exact same function `train_pipeline.main()` runs in production, so results are fully reproducible via the CLI script.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_style("whitegrid")

ML_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = ML_ROOT / "src"
sys.path.append(str(SRC_PATH))

from train_pipeline import (
    prepare_data,
    compare_models,
    tune_and_fit_best_model,
    write_comparison_report,
    MODEL_OUTPUT_PATH,
    TEST_SIZE,
    CV_FOLDS,
    RANDOM_STATE,
)

print(f"Test size: {TEST_SIZE} | CV folds: {CV_FOLDS} | Random state: {RANDOM_STATE}")

## 1. Data Preparation

Runs cleaning + feature engineering, then produces a stratified 80/20 split preserving the default/no-default ratio in both sets.

In [ ]:
X_train, y_train, X_test, y_test = prepare_data()

print(f"Train shape: {X_train.shape} | Default rate: {y_train.mean():.4f}")
print(f"Test shape:  {X_test.shape} | Default rate: {y_test.mean():.4f}")

## 2. Candidate Model Comparison

Each candidate — Logistic Regression (baseline/interpretable), Random Forest, and XGBoost — is evaluated on identical 5-fold stratified cross-validation using Accuracy, Precision, Recall, F1, and ROC-AUC. Class imbalance is handled via `class_weight="balanced"` / `scale_pos_weight`.

In [ ]:
start = time.time()
comparison_df, candidates = compare_models(X_train, y_train)
print(f"Comparison completed in {time.time() - start:.1f}s\n")
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
melted = comparison_df.melt(id_vars="model",
                             value_vars=["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean"],
                             var_name="metric", value_name="score")
sns.barplot(data=melted, x="model", y="score", hue="metric", ax=ax)
ax.set_title("Candidate Model Comparison — Cross-Validated Metrics")
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. Model Selection

The model is selected primarily by **recall on the Default class**, then ROC-AUC as a tiebreaker — per the business requirement that missing a true defaulter is costlier than a false alarm.

In [ ]:
best_model_name = comparison_df.iloc[0]["model"]
print(f"Selected best model family: {best_model_name}")
print(f"CV recall (Default class): {comparison_df.iloc[0]['recall_mean']:.4f}")
print(f"CV ROC-AUC: {comparison_df.iloc[0]['roc_auc_mean']:.4f}")

## 4. Hyperparameter Tuning

`RandomizedSearchCV` searches the hyperparameter space for the selected model family, optimizing recall on the Default class, then refits the full `Pipeline` (preprocessor + tuned model) on the entire training set.

In [ ]:
best_pipeline, search_summary = tune_and_fit_best_model(
    best_model_name, candidates, X_train, y_train
)

print(f"Best model: {search_summary['best_model']}")
print(f"Best CV recall: {search_summary['best_cv_recall']:.4f}")
print("Best hyperparameters:")
for k, v in search_summary["best_params"].items():
    print(f"  {k}: {v}")

## 5. Save the Final Pipeline

The tuned model and its full preprocessing chain are already wrapped into a single `sklearn.pipeline.Pipeline`. Serialize it with `joblib.dump` — this is the exact artifact FastAPI's `model_loader.py` will load at startup.

In [ ]:
MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, MODEL_OUTPUT_PATH)
print(f"Pipeline saved to: {MODEL_OUTPUT_PATH}")
print(f"Pipeline steps: {[name for name, _ in best_pipeline.steps]}")

## 6. Write Model Comparison Report

In [ ]:
write_comparison_report(comparison_df, search_summary, X_train, X_test)
print("Report written to ml/reports/model_comparison_report.md")

## 7. Summary

- All three candidates (Logistic Regression, Random Forest, XGBoost) were compared under identical 5-fold stratified cross-validation.
- The model family with the strongest recall on the Default class was selected and tuned via `RandomizedSearchCV`.
- The final tuned `Pipeline` — preprocessing (imputers, scaler, one-hot encoder) plus the tuned estimator — was serialized to `ml/models/loan_pipeline.pkl` via `joblib.dump`, ready for FastAPI to load as a singleton at startup.
- Held-out `train_set.csv` / `test_set.csv` splits were persisted so `04_model_evaluation.ipynb` (and `ml/src/evaluate_model.py`) can independently score the saved artifact.

**Next step:** `04_model_evaluation.ipynb` — confusion matrix, classification report, ROC-AUC, and feature importance on the held-out test set.